# Deep Learning: offline companion notebook
Run in a clean kernel. See SETUP.md. Read corresponding lessons and vary one parameter after each example.


## D01 — autograd mlp
Dependencies: torch.


In [ ]:
import torch
x = torch.tensor([1.,2.,3.],requires_grad=True)
(x.square().sum()).backward()
assert torch.allclose(x.grad,2*x)
torch.manual_seed(7)
X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y = torch.tensor([[0.],[1.],[1.],[0.]])
model = torch.nn.Sequential(torch.nn.Linear(2,8),torch.nn.Tanh(),torch.nn.Linear(8,1))
opt = torch.optim.AdamW(model.parameters(),lr=.05)
for _ in range(250):
    opt.zero_grad(); loss = torch.nn.functional.binary_cross_entropy_with_logits(model(X),y); loss.backward(); opt.step()
with torch.no_grad(): print(torch.sigmoid(model(X)))

## D02 — train eval
Dependencies: torch.


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(1)
X=torch.randn(160,4); y=(X.sum(1)>0).long()
train=DataLoader(TensorDataset(X[:120],y[:120]),batch_size=16,shuffle=True)
model=torch.nn.Sequential(torch.nn.Linear(4,16),torch.nn.ReLU(),torch.nn.Dropout(.1),torch.nn.Linear(16,2))
opt=torch.optim.AdamW(model.parameters(),lr=.01)
for epoch in range(5):
    model.train()
    for xb,yb in train:
        opt.zero_grad(); loss=torch.nn.functional.cross_entropy(model(xb),yb); loss.backward(); opt.step()
    model.eval()
    with torch.inference_mode(): print(epoch,(model(X[120:]).argmax(1)==y[120:]).float().mean().item())

## D03 — causal attention
Dependencies: torch.


In [ ]:
import torch
from math import sqrt
def causal_attention(q,k,v):
    scores=q@k.transpose(-1,-2)/sqrt(q.shape[-1])
    mask=torch.ones(scores.shape[-2:],dtype=torch.bool).triu(1)
    return scores.masked_fill(mask,float('-inf')).softmax(-1)@v
x=torch.randn(3,4); changed=x.clone(); changed[2]+=100
assert torch.allclose(causal_attention(x,x,x)[:2],causal_attention(changed,changed,changed)[:2])
print(causal_attention(x,x,x).shape)

## D04 — diffusion noising
Dependencies: torch.


In [ ]:
import torch
torch.manual_seed(3)
x=torch.ones(1000); eps=torch.randn_like(x)
for alpha_bar in (.99,.5,.01):
    noisy=alpha_bar**.5*x+(1-alpha_bar)**.5*eps
    print(alpha_bar, (noisy-x).square().mean().item())
# Forward process only; reverse denoising requires a learned predictor.